In [1]:
import requests
import fitz
from bs4 import BeautifulSoup
from openai import OpenAI
import os
import datetime
from dotenv import load_dotenv

# Load env variables
load_dotenv()

# Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)
LLM_MODEL = "gpt-4o-mini"
OUTPUT_DIR = "deepdives"

def fetch_paper_text(url):
    """
    Downloads a paper from a given URL (handling ArXiv/Landing pages) 
    and extracts text using PyMuPDF.
    Returns: (text, title_from_metadata)
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    print(f"📥 Fetching: {url}")
    pdf_url = url
    
    try:
        # 1. Handle ArXiv Abstracts -> PDF
        if "arxiv.org/abs" in url:
            pdf_url = url.replace("/abs/", "/pdf/")
            if not pdf_url.endswith(".pdf"): pdf_url += ".pdf"
            
        # 2. Handle Generic Landing Pages (if not ending in .pdf)
        elif not url.lower().endswith(".pdf"):
            try:
                response = requests.get(url, headers=headers, timeout=10)
                soup = BeautifulSoup(response.text, "html.parser")
                # Look for a link containing "pdf" or "download"
                found_link = soup.find('a', href=True, string=lambda t: t and ("pdf" in t.lower() or "download" in t.lower()))
                # Fallback: check href attributes
                if not found_link:
                    found_link = soup.find('a', href=lambda h: h and ".pdf" in h.lower())
                
                if found_link:
                    href = found_link['href']
                    if href.startswith("http"):
                        pdf_url = href
                    elif href.startswith("/"):
                        # Reconstruct base url
                        from urllib.parse import urlparse
                        parsed = urlparse(url)
                        pdf_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                    else:
                        pdf_url = url.rstrip("/") + "/" + href
                    print(f"   -> Detected PDF Link: {pdf_url}")
            except Exception as e:
                print(f"   ⚠️ Could not scrape landing page: {e}. Trying original URL.")

        # 3. Download PDF
        response = requests.get(pdf_url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # 4. Extract Text & Metadata
        with fitz.open(stream=response.content, filetype="pdf") as doc:
            text = ""
            # Extract metadata title if available, else use filename/URL
            meta_title = doc.metadata.get('title', '')
            if not meta_title or meta_title.strip() == "":
                meta_title = url.split("/")[-1]
            
            # Read first 25 pages (enough for methodology/results, skips huge appendix)
            for page in doc[:25]:
                text += page.get_text()
                
        return text, meta_title

    except Exception as e:
        return f"[ERROR] Failed to fetch: {e}", "Unknown Title"

def generate_deep_dive(text, title, original_url):
    """
    Sends the text to the LLM for technical analysis.
    """
    if text.startswith("[ERROR]"):
        return f"## Error processing {original_url}\n{text}\n"

    prompt = f"""
You are a highly specialized expert curator for **“ets4 Monthly (Economic Time Series Forecasting Monthly),”** a newsletter focused exclusively on **practical and impactful forecasting of economic time series.** 
You are also a seasoned **econometrician, forecaster, and data scientist** with deep experience evaluating new modeling techniques, understanding their assumptions and limits, and interpreting empirical evidence.

Your task is to produce a **technical Deep Dive** into the following research paper.

Before producing your analysis:

• **Introduce the foundational concepts** necessary to understand the paper’s contribution.
  – Provide a *brief informal explanation* (intuition first).  
  – Provide a *concise formal explanation* (a few key equations only).  
  – These foundations may be broader than the specific innovation, and should orient a researcher not specialized in this sub-field.

• When analyzing the core idea of the paper, **focus on the intuition, design choices, assumptions, and where the method works or fails**, rather than long derivations.  
  – Highlight examples, counterexamples, and scenarios where the method is brittle or where assumptions are unrealistic.

• Provide **two–three sentences on the literature gap** the paper fills and why it matters.

• Read empirical sections *critically*:  
  – Extract findings from tables and figures.  
  – Pay attention to situations where the method *underperforms*, including in Monte Carlo experiments (and whether the MC design is sound).  
  – Emphasize practical implications and caveats for real-world forecasters.

---

## **Your Output Structure (strictly follow this):**

1. **Foundational Concepts (Informal + Formal):**  
   The minimum background a non-specialist economist/forecaster needs to follow the paper. Include 1–3 key formulas max.

2. **The Core Innovation:**  
   The specific modeling contribution. Explain the intuition, what it tries to fix, the assumptions under which it works, and importantly **when and why it may fail**.

3. **Methodology:**  
   Model architecture, estimation strategy, feature engineering, experimental setup. Keep it clear and emphasize design decisions over math. Include 1–3 key formulas max.

4. **Position in the Literature:**  
   Briefly state (2–3 sentences) what gap the paper fills.

5. **Empirical Evidence:**  
   Dataset(s), forecasting setup, evaluation metrics, performance vs. baselines.  
   Highlight **both strengths and weak points**, especially where tables/figures reveal weaknesses or instability.

6. **Critical Takeaway (for Practitioners):**  
   One sentence on why this paper matters—or why it doesn’t—for real-world economic time-series forecasting.

---

**TEXT START**
{text[:80000]} 
**TEXT END**
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM Error: {e}"

def deepdive(links_vector):
    """
    Takes a list of strings (URLs), downloads them, analyzes them, 
    and saves a report in the 'deepdives' folder with today's date.
    """
    # 1. Setup Directory and Filename
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today_str = datetime.date.today().strftime('%Y-%m-%d')
    filename = f"ets4_deepdive_monthly_{today_str}.md"
    output_path = os.path.join(OUTPUT_DIR, filename)
    
    report_content = [f"# Deep Dive Report - {today_str}\n"]
    report_content.append(f"Analyzing {len(links_vector)} papers.\n")
    
    # 2. Process Links
    for i, link in enumerate(links_vector, 1):
        print(f"\n--- Processing Paper {i}/{len(links_vector)} ---")
        
        # Get Text
        full_text, title = fetch_paper_text(link)
        
        # Analyze
        if len(full_text) > 500:
            print(f"   -> Analyzing '{title}' with LLM...")
            analysis = generate_deep_dive(full_text, title, link)
            
            # Format Output
            section = f"## {i}. {title}\n**Source:** {link}\n\n{analysis}\n\n---\n"
            report_content.append(section)
        else:
            print("   -> ⚠️ Text too short or extraction failed.")
            report_content.append(f"## {i}. {link}\n\n*Extraction failed or PDF was empty.*\n\n---\n")

    # 3. Save to specific path
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(report_content))
    
    print(f"\n✅ Report generated successfully at: {output_path}")

In [2]:
# ------------- MAIN ----------------

if __name__ == "__main__":
    
    # PUT YOUR LINKS HERE
    my_links = [
        "https://arxiv.org/pdf/2511.07014",
        "https://www.arxiv.org/abs/2511.07678",
    ]

    deepdive(my_links)


--- Processing Paper 1/2 ---
📥 Fetching: https://arxiv.org/pdf/2511.07014
   -> Analyzing 'Diffolio: A Diffusion Model for Multivariate Probabilistic Financial Time-Series Forecasting and Portfolio Construction' with LLM...

--- Processing Paper 2/2 ---
📥 Fetching: https://www.arxiv.org/abs/2511.07678
   -> Analyzing 'AIA Forecaster: Technical Report' with LLM...

✅ Report generated successfully at: deepdives/ets4_deepdive_monthly_2025-11-24.md
